# Integrated Manual Curation Workflow

This notebook merges `manual_correction.ipynb` and `manual_correction2.ipynb` into a **single phase-driven workflow**.

Key principles:
- Preserve the original function logic from both notebooks.
- Reorganize functions into Phases 1-5 for reproducible execution.
- Add clear English comments/doc guidance for each functional block.
- Do not auto-run any phase by default.

In [1]:
from __future__ import annotations

import os
import re
import glob
import json
import time
import subprocess
from pathlib import Path

import requests
import pandas as pd
from Bio import SeqIO

import cobra
from cobra.flux_analysis import flux_variability_analysis, gapfilling
from cobra.io import load_json_model

In [ ]:
# Shared path/config variables reused by original functions.
# Existing logic is preserved; only organization is improved.

TARGET_MODELS = [
    'iML1515',
]

DOWNLOAD_DIR = 'bigg_resources'
MODELS_DIR = os.path.join(DOWNLOAD_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

STATIC_MODELS_URL = 'http://bigg.ucsd.edu/static/models'
UNIPROT_API = 'https://rest.uniprot.org/uniprotkb/{}.fasta'

DIR = 'bigg_resources'
json_main = os.path.join(DIR, 'global_gene_to_rxn.json')
json_part = os.path.join(DIR, 'global_gene_to_rxn_1.json')
fasta_main = os.path.join(DIR, 'bigg_proteins.fasta')
fasta_part = os.path.join(DIR, 'bigg_proteins_1.fasta')
fasta_temp = os.path.join(DIR, 'temp_merged.fasta')

OUTPUT_ROOT = Path('manual_curation_outputs_merged')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Integrated notebook ready. Output root: {OUTPUT_ROOT.resolve()}')

## Phase 1: Genome-to-Network Primary Mapping (Bottom-Up Foundation)
Source: File 1

In [ ]:
# ---------------------------------------------------------------------
# Function: download_target_models
# Purpose: Download selected BiGG models as local references.
# ---------------------------------------------------------------------
def download_target_models():
    """translated_note"""
    print(f">>> translated_note: {TARGET_MODELS}")
    for model_id in TARGET_MODELS:
        json_path = os.path.join(MODELS_DIR, f"{model_id}.json")
        if os.path.exists(json_path):
            print(f"  - {model_id} translated_note, translated_note. ")
            continue
            
        url = f"{STATIC_MODELS_URL}/{model_id}.json"
        res = requests.get(url)
        if res.status_code == 200:
            with open(json_path, "wb") as f:
                f.write(res.content)
            print(f"  -  {model_id} translated_note!")
        else:
            print(f"  -  {model_id} translated_note!")


# ---------------------------------------------------------------------
# Function: parse_and_fetch_sequences
# Purpose: Build BiGG protein FASTA and gene-to-reaction mapping from annotations.
# ---------------------------------------------------------------------
def parse_and_fetch_sequences():
    """translated_note, translated_note UniProt translated_note"""
    print("\n>>> translated_note...")
    
    gene_to_rxn = {}
    fasta_out_path = os.path.join(DOWNLOAD_DIR, "bigg_proteins.fasta")
    
    # translated_note, translated_note
    downloaded_genes = set()
    if os.path.exists(fasta_out_path):
        with open(fasta_out_path, "r") as f:
            for line in f:
                if line.startswith(">"):
                    downloaded_genes.add(line.strip()[1:])

    with open(fasta_out_path, "a") as fasta_file:
        for model_id in TARGET_MODELS:
            filepath = os.path.join(MODELS_DIR, f"{model_id}.json")
            if not os.path.exists(filepath): continue
            
            with open(filepath, "r") as f:
                model_data = json.load(f)
            
            # 1. translated_note Gene -> Rxn translated_note
            for rxn in model_data.get("reactions", []):
                rxn_id = rxn.get("id")
                genes = rxn.get("gene_reaction_rule", "").replace("and", "").replace("or", "").replace("(", "").replace(")", "").split()
                for gene in genes:
                    if not gene: continue
                    global_id = f"{model_id}_{gene}"
                    if global_id not in gene_to_rxn:
                        gene_to_rxn[global_id] = set()
                    gene_to_rxn[global_id].add(rxn_id)
            
            # 2. translated_note UniProt translated_note
            print(f"\n--- translated_note {model_id} translated_note ---")
            for gene_info in model_data.get("genes", []):
                global_id = f"{model_id}_{gene_info.get('id')}"
                if global_id in downloaded_genes:
                    continue
                
                annotations = gene_info.get("annotation", {})
                # translated_note UniProt
                if "uniprot" in annotations:
                    uid = annotations["uniprot"][0]
                    res = requests.get(UNIPROT_API.format(uid))
                    if res.status_code == 200:
                        seq_lines = res.text.strip().split('\n')
                        sequence = "".join(seq_lines[1:])
                        
                        fasta_file.write(f">{global_id}\n")
                        for i in range(0, len(sequence), 80):
                            fasta_file.write(sequence[i:i+80] + "\n")
                        fasta_file.flush()
                        print(f"  [+] translated_note: {global_id} (UniProt: {uid})")
                    else:
                        print(f"  [-] translated_note: {global_id} (API translated_note)")
                    time.sleep(0.2) # translated_note

    # translated_note
    mapping_file = os.path.join(DOWNLOAD_DIR, "global_gene_to_rxn.json")
    with open(mapping_file, "w") as f:
        json.dump({k: list(v) for k, v in gene_to_rxn.items()}, f, indent=2)
    print(f"\n translated_note: {mapping_file}")


# ---------------------------------------------------------------------
# Function: merge_json_files
# Purpose: Merge split gene-to-reaction mapping JSON files.
# ---------------------------------------------------------------------
def merge_json_files():
    print(">>> translated_note JSON translated_note...")
    merged_dict = {}
    
    # translated_note JSON
    if os.path.exists(json_main):
        with open(json_main, "r") as f:
            merged_dict = json.load(f)
            
    # translated_note JSON translated_note
    if os.path.exists(json_part):
        with open(json_part, "r") as f:
            dict_2 = json.load(f)
            for gene, rxns in dict_2.items():
                if gene in merged_dict:
                    # translated_note
                    merged_dict[gene] = list(set(merged_dict[gene] + rxns))
                else:
                    merged_dict[gene] = rxns
                    
    # translated_note JSON translated_note
    with open(json_main, "w") as f:
        json.dump(merged_dict, f, indent=2)
    print(f" JSON translated_note!translated_note {len(merged_dict)} translated_note. ")


# ---------------------------------------------------------------------
# Function: merge_fasta_files
# Purpose: Merge and deduplicate FASTA sequence libraries.
# ---------------------------------------------------------------------
def merge_fasta_files():
    print("\n>>> translated_note FASTA translated_note...")
    seen_headers = set()
    count = 0
    
    with open(fasta_temp, "w") as f_out:
        # translated_note fasta
        def process_fasta(filepath):
            nonlocal count
            if not os.path.exists(filepath): return
            with open(filepath, "r") as f_in:
                write_flag = False
                for line in f_in:
                    if line.startswith(">"):
                        header = line.strip()
                        if header not in seen_headers:
                            seen_headers.add(header)
                            write_flag = True
                            f_out.write(line)
                            count += 1
                        else:
                            write_flag = False  # translated_note, translated_note
                    else:
                        if write_flag:
                            f_out.write(line)
                            
        process_fasta(fasta_main)
        process_fasta(fasta_part)
        
    # translated_note FASTA translated_note
    os.replace(fasta_temp, fasta_main)
    print(f" FASTA translated_note!translated_note {count} translated_note. ")


# ---------------------------------------------------------------------
# Function: cleanup
# Purpose: Remove temporary split files after merge.
# ---------------------------------------------------------------------
def cleanup():
    print("\n>>> translated_note...")
    if os.path.exists(json_part):
        os.remove(json_part)
        print(f"  - translated_note {json_part}")
    if os.path.exists(fasta_part):
        os.remove(fasta_part)
        print(f"  - translated_note {fasta_part}")
    print(" translated_note!translated_note. ")


# ---------------------------------------------------------------------
# Function: create_fasta_from_gbff
# Purpose: Export reference CDS translations from GenBank to FASTA.
# ---------------------------------------------------------------------
def create_fasta_from_gbff(gbff_file="genomes/Rpal_BisA53.gb", fasta_out="prots/Rpal_ref.fasta"):
    print(f"\n[1/4] translated_note {gbff_file} translated_note FASTA...")
    records = []
    try:
        for rec in SeqIO.parse(gbff_file, "genbank"):
            for feat in rec.features:
                if feat.type == "CDS":
                    seq = feat.qualifiers.get("translation", [""])[0]
                    locus_tag = feat.qualifiers.get("locus_tag", [""])[0]
                    if seq and locus_tag:
                        records.append(f">{locus_tag}\n{seq}\n")
        with open(fasta_out, "w") as f:
            f.writelines(records)
        print(f" translated_note {len(records)} translated_note {fasta_out}")
    except FileNotFoundError:
        print(f" translated_note {gbff_file}")
        exit()


# ---------------------------------------------------------------------
# Function: run_bbh_pipeline
# Purpose: Run strict reciprocal BLASTP (BBH) mapping.
# ---------------------------------------------------------------------
def run_bbh_pipeline(ref_fasta="Rpal_ref.fasta", target_fasta="PGIDNB_proteome.fasta", e_value="1e-10"):
    print(f"\n[2/4] translated_note BLAST+ translated_note (BBH)...")
    
    # translated_note
    print("  -> translated_note BLAST translated_note...")
    subprocess.run(["makeblastdb", "-in", ref_fasta, "-dbtype", "prot", "-out", "db_ref"], stdout=subprocess.DEVNULL)
    subprocess.run(["makeblastdb", "-in", target_fasta, "-dbtype", "prot", "-out", "db_target"], stdout=subprocess.DEVNULL)
    
    # translated_note BLAST (translated_note Top 1)
    print("  -> translated_note Ref -> Target translated_note...")
    subprocess.run(["blastp", "-query", ref_fasta, "-db", "db_target", "-outfmt", "6 qseqid sseqid pident evalue", 
                    "-evalue", e_value, "-max_target_seqs", "1", "-out", "r2t.tsv"])
                    
    print("  -> translated_note Target -> Ref translated_note...")
    subprocess.run(["blastp", "-query", target_fasta, "-db", "db_ref", "-outfmt", "6 qseqid sseqid pident evalue", 
                    "-evalue", e_value, "-max_target_seqs", "1", "-out", "t2r.tsv"])
    
    # translated_note, translated_note BBH
    print("  -> translated_note...")
    r2t = pd.read_csv("r2t.tsv", sep="\t", header=None, names=["ref", "target", "pident1", "e1"])
    t2r = pd.read_csv("t2r.tsv", sep="\t", header=None, names=["target", "ref", "pident2", "e2"])
    
    # translated_note Best Hit
    dict_r2t = r2t.set_index('ref')['target'].to_dict()
    dict_t2r = t2r.set_index('target')['ref'].to_dict()
    
    bbh_pairs = []
    for r_gene, t_gene in dict_r2t.items():
        if dict_t2r.get(t_gene) == r_gene:
            bbh_pairs.append({"Ref_Gene": r_gene, "Target_Gene": t_gene})
            
    bbh_df = pd.DataFrame(bbh_pairs)
    bbh_df.to_csv("bbh_mapping_result.csv", index=False)
    print(f" BBH translated_note!translated_note {len(bbh_pairs)} translated_note, translated_note bbh_mapping_result.csv")
    return dict_r2t, dict_t2r


# ---------------------------------------------------------------------
# Function: update_model_genes
# Purpose: Rewrite model GPR gene IDs using BBH and remove orphan reactions.
# ---------------------------------------------------------------------
def update_model_genes(model_file="purple_biology_strict.json", bbh_csv="bbh_mapping_result.csv", out_file="purple_genes_cleaned.json"):
    print(f"\n[3/4] translated_note...")
    model = cobra.io.load_json_model(model_file)
    bbh_df = pd.read_csv(bbh_csv)
    # translated_note Ref -> Target translated_note
    mapping_dict = bbh_df.set_index("Ref_Gene")["Target_Gene"].to_dict()
    
    replaced_count = 0
    orphaned_reactions = []
    
    for rxn in model.reactions:
        rule = rxn.gene_reaction_rule
        if not rule: continue
        
        # translated_note RPE_RS translated_note
        import re
        suspects = re.findall(r'RPE_RS\d+', rule)
        if not suspects: continue
        
        new_rule = rule
        all_mapped = True
        
        for rpe in suspects:
            if rpe in mapping_dict:
                # translated_note BBH, translated_note!
                target_gene = mapping_dict[rpe]
                new_rule = new_rule.replace(rpe, target_gene)
                replaced_count += 1
            else:
                # translated_note BBH, translated_note
                all_mapped = False
        
        if all_mapped:
            rxn.gene_reaction_rule = new_rule
        else:
            orphaned_reactions.append(rxn)
            
    print(f" translated_note {replaced_count} translated_note ID!")
    
    print(f"\n[4/4] translated_note!translated_note...")
    print(f" translated_note {len(orphaned_reactions)} translated_note RPE_RS translated_note!")
    
    # translated_note, translated_note
    for rxn in orphaned_reactions:
        print(f"   - translated_note: {rxn.id} | translated_note: {rxn.gene_reaction_rule}")
        model.remove_reactions([rxn])
        
    cobra.io.save_json_model(model, out_file)
    print(f"\n translated_note: {out_file} (translated_note!)")


## Phase 2: Topology Cleanup & Macromolecule Standardization (Network Stitching)
Source: File 1 + File 2

In [ ]:
# ---------------------------------------------------------------------
# Function: update_model_with_conflicts
# Purpose: Resolve GPR conflicts and apply safe single-gene replacement from BiGG.
# ---------------------------------------------------------------------
def update_model_with_conflicts(update_model_file, source_models_dir, conflicts_file, out_model_file):
    print(">>> [FBA Agent] translated_note GPR translated_note (translated_note)...")

    # 1. translated_note
    print(f"  - translated_note: {update_model_file}")
    with open(update_model_file, 'r') as f:
        target_model = json.load(f)
        
    # 2. translated_note (translated_note)
    source_rxns = {}
    source_mets = {}
    
    model_files = glob.glob(os.path.join(source_models_dir, '*.json'))
    if not model_files:
        print(f" translated_note: translated_note '{source_models_dir}' translated_note .json translated_note. ")
        return
        
    print(f"  - translated_note {len(model_files)} translated_note BiGG translated_note...")
    for m_file in model_files:
        with open(m_file, 'r', encoding='utf-8') as f:
            try:
                s_model = json.load(f)
                # translated_note (BiGG ID translated_note, translated_note)
                for r in s_model.get('reactions', []):
                    if r['id'] not in source_rxns:
                        source_rxns[r['id']] = r
                for m in s_model.get('metabolites', []):
                    if m['id'] not in source_mets:
                        source_mets[m['id']] = m
            except Exception as e:
                print(f"     translated_note {m_file} translated_note, translated_note: {e}")

    print(f"  - translated_note: translated_note {len(source_rxns)} translated_note, {len(source_mets)} translated_note. ")

    # ---------------------------------------------------------
    # translated_note: translated_note "_u" translated_note Compartment translated_note
    # ---------------------------------------------------------
    print("  - translated_note '_u' translated_note...")
    for met in target_model.get("metabolites", []):
        if met["id"].endswith("_u"):
            met["id"] = met["id"][:-2]  # translated_note _u
            comp_match = re.search(r'_([a-zA-Z0-9]+)$', met["id"])
            if comp_match:
                met["compartment"] = comp_match.group(1)

    for rxn in target_model.get("reactions", []):
        new_met_dict = {}
        for met_id, stoich in rxn.get("metabolites", {}).items():
            if met_id.endswith("_u"):
                new_met_dict[met_id[:-2]] = stoich
            else:
                new_met_dict[met_id] = stoich
        rxn["metabolites"] = new_met_dict

    # ---------------------------------------------------------
    # translated_note: translated_note CSV translated_note
    # ---------------------------------------------------------
    print(f"  - translated_note: {conflicts_file}")
    try:
        # translated_note, translated_note UnicodeDecodeError
        try:
            df_conflicts = pd.read_csv(conflicts_file, encoding='utf-8')
        except UnicodeDecodeError:
            print("    ! translated_note: translated_note, translated_note 'gbk' translated_note...")
            try:
                df_conflicts = pd.read_csv(conflicts_file, encoding='gbk')
            except UnicodeDecodeError:
                print("    ! translated_note: translated_note 'latin1' translated_note...")
                df_conflicts = pd.read_csv(conflicts_file, encoding='latin1')
    except FileNotFoundError:
        print(f" translated_note: translated_note {conflicts_file}. translated_note. ")
        return

    # ---------------------------------------------------------
    # translated_note: translated_note GPR translated_note, translated_note, translated_note
    # ---------------------------------------------------------
    manual_review_cases = []
    auto_replaced_count = 0
    missing_in_bigg = 0
    
    # translated_note ID translated_note, translated_note
    target_rxn_ids = {r['id'] for r in target_model.get('reactions', [])}
    target_met_ids = {m['id'] for m in target_model.get('metabolites', [])}

    print(f"  - translated_note {len(df_conflicts)} translated_note...")
    
    for _, row in df_conflicts.iterrows():
        target_gene = str(row['Model_Gene']).strip()
        old_rxns = str(row['Model_Claimed_Rxns']).split(' | ') if pd.notna(row['Model_Claimed_Rxns']) else []
        new_rxns = str(row['BiGG_True_Rxns']).split(' | ') if pd.notna(row['BiGG_True_Rxns']) else []
        
        for old_rxn_id in old_rxns:
            old_rxn_id = old_rxn_id.strip()
            # translated_note
            rxn_index = next((i for i, r in enumerate(target_model['reactions']) if r['id'] == old_rxn_id), None)
            if rxn_index is None:
                continue
                
            rxn_obj = target_model['reactions'][rxn_index]
            gpr = rxn_obj.get('gene_reaction_rule', '')
            
            # translated_note
            genes_in_rule = [g.strip() for g in gpr.replace('and', '').replace('or', '').replace('(', '').replace(')', '').split() if g.strip()]
            
            # translated_note: translated_note, translated_note
            if len(genes_in_rule) == 1 and genes_in_rule[0] == target_gene:
                # translated_note
                del target_model['reactions'][rxn_index]
                
                # translated_note
                for new_rxn_id in new_rxns:
                    new_rxn_id = new_rxn_id.strip()
                    if new_rxn_id in source_rxns:
                        if new_rxn_id not in target_rxn_ids:
                            new_rxn_obj = source_rxns[new_rxn_id].copy()
                            # translated_note GPR translated_note
                            new_rxn_obj['gene_reaction_rule'] = target_gene
                            target_model['reactions'].append(new_rxn_obj)
                            target_rxn_ids.add(new_rxn_id)
                            
                            # translated_note
                            for met_id in new_rxn_obj.get('metabolites', {}):
                                if met_id in source_mets and met_id not in target_met_ids:
                                    target_model['metabolites'].append(source_mets[met_id])
                                    target_met_ids.add(met_id)
                    else:
                        missing_in_bigg += 1
                                
                auto_replaced_count += 1
            else:
                # translated_note: translated_note/translated_note, translated_note, translated_note
                manual_review_cases.append({
                    "Target_Gene": target_gene,
                    "Conflict_Reaction": old_rxn_id,
                    "Current_GPR": gpr,
                    "Suggested_BiGG_Rxns": " | ".join(new_rxns),
                    "Action_Needed": "Manual Review (Multi-gene complex or isozymes)"
                })
                
    # ---------------------------------------------------------
    # translated_note: translated_note
    # ---------------------------------------------------------
    if manual_review_cases:
        df_manual = pd.DataFrame(manual_review_cases)
        manual_file = "gpr_manual_review_required.csv"
        df_manual.to_csv(manual_file, index=False, encoding='utf-8-sig') # translated_note utf-8-sig translated_note Excel translated_note
        print(f"\n translated_note {len(manual_review_cases)} translated_note, translated_note: {manual_file}")
        print("   (translated_note, translated_note)")
        
    print(f"\n translated_note {auto_replaced_count} translated_note. translated_note. ")
    if missing_in_bigg > 0:
        print(f" translated_note: translated_note {missing_in_bigg} translated_note, translated_note. ")
    
    with open(out_model_file, 'w', encoding='utf-8') as f:
        json.dump(target_model, f, indent=4)
    print(f">>> translated_note v2 translated_note: {out_model_file}")


# ---------------------------------------------------------------------
# Function: clean_model_and_run_fba
# Purpose: Merge duplicate metabolites and export orphan diagnostics.
# ---------------------------------------------------------------------
def clean_model_and_run_fba(in_model_file="updated_consensus_v2.json", out_model_file="updated_consensus_v3.json"):
    print(">>> [FBA Agent] translated_note...")
    model = cobra.io.load_json_model(in_model_file)
    
    # ==========================================
    # translated_note 1: translated_note (Orphans)
    # ==========================================
    orphan_data = []
    for met in model.metabolites:
        is_boundary = any(r.boundary for r in met.reactions)
        # translated_note, translated_note
        if len(met.reactions) == 1 and not is_boundary:
            orphan_data.append({
                "Metabolite_ID": met.id,
                "Name": met.name,
                "Formula": met.formula,
                "Compartment": met.compartment,
                "Reaction_Involved": list(met.reactions)[0].id
            })
            
    df_orphans = pd.DataFrame(orphan_data)
    orphan_file = "model_orphans_report.csv"
    df_orphans.to_csv(orphan_file, index=False, encoding="utf-8-sig")
    print(f"   translated_note {len(df_orphans)} translated_note: {orphan_file}")

    # ==========================================
    # translated_note 2: translated_note (Merge Duplicates)
    # ==========================================
    # translated_note: {(compartment, formula, normalized_name): [met_obj1, met_obj2...]}
    group_dict = {}
    for met in model.metabolites:
        if met.formula and met.compartment:
            key = (met.compartment, met.formula, normalize_name(met.name))
            group_dict.setdefault(key, []).append(met)
            
    merge_count = 0
    for key, mets in group_dict.items():
        if len(mets) > 1:
            # translated_note ID translated_note, translated_note, translated_note ID(translated_note 2pglyc_c, translated_note 2pglyc_cx_c)
            mets = sorted(mets, key=lambda x: len(x.id))
            target_met = mets[0]
            
            for duplicate_met in mets[1:]:
                # translated_note
                # translated_note list() translated_note
                for rxn in list(duplicate_met.reactions):
                    # translated_note (stoichiometry)
                    stoich = rxn.metabolites[duplicate_met]
                    # translated_note: translated_note target_met, translated_note duplicate_met
                    rxn.add_metabolites({target_met: stoich, duplicate_met: -stoich}, combine=True)
                
                # translated_note
                model.remove_metabolites([duplicate_met])
                merge_count += 1
                print(f"   translated_note: {duplicate_met.id} translated_note -> {target_met.id} ({target_met.name})")

    print(f"\n translated_note!translated_note {merge_count} translated_note. ")
    
    # translated_note v3 translated_note
    cobra.io.save_json_model(model, out_model_file)
    print(f"   translated_note: {out_model_file}")

    # ==========================================
    # translated_note 3: translated_note FBA translated_note
    # ==========================================
    print("\n>>> translated_note FBA translated_note...")
    try:
        solution = model.optimize()
        print("\n================ FBA translated_note ================")
        print(f" translated_note: {solution.status}")
        print(f" translated_note (Objective Value): {solution.objective_value:.4f} h^-1")
        print("==============================================\n")
        
        if solution.objective_value < 1e-6:
            print("")
        else:
            print("")
            
    except Exception as e:
        print(f" FBA translated_note: {e}")


# ---------------------------------------------------------------------
# Function: unify_quinolinate_ids
# Purpose: Normalize quinolinate-related IDs (e.g., qns_c -> quln_c).
# ---------------------------------------------------------------------
def unify_quinolinate_ids(model_file="updated_consensus_purple.json", out_file="updated_consensus_v5_fixed.json"):
    print(">>> [FBA Agent] translated_note Quinolinate (quln_c) ID translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    # 1. translated_note: translated_note NAD translated_note quln_c
    target_id = "quln_c"
    # translated_note ID translated_note
    alias_ids = ["qns_c", "quinolinate_c", "cpd00371"] # translated_note quin_c (translated_note)

    if target_id not in model.metabolites:
        print(f" translated_note: translated_note ID {target_id}!")
        return

    target_met = model.metabolites.get_by_id(target_id)
    
    # 2. translated_note
    for alias in alias_ids:
        if alias in model.metabolites:
            alias_met = model.metabolites.get_by_id(alias)
            print(f"   translated_note {alias}, translated_note {target_id}...")
            for rxn in list(alias_met.reactions):
                coeff = rxn.metabolites[alias_met]
                rxn.add_metabolites({target_met: coeff, alias_met: -coeff}, combine=True)
            model.remove_metabolites([alias_met])

    # 3. translated_note NNDPR (Quinolinate -> NaMN)
    # translated_note quln_c, translated_note, translated_note
    print("\n translated_note NNDPR (translated_note)...")
    found_nndpr = False
    for rxn in model.reactions:
        if "NNDPR" in rxn.id or "quinolinate phosphoribosyltransferase" in rxn.name.lower():
            found_nndpr = True
            # translated_note quln_c translated_note prpp_c, translated_note nicrnt_c
            # translated_note, translated_note
            print(f"   translated_note NNDPR translated_note: {rxn.id} | translated_note: {rxn.reaction}")
            
            # translated_note (translated_note ID translated_note)
            correct_stoich = {
                model.metabolites.get_by_id("quln_c"): -1.0,
                model.metabolites.get_by_id("prpp_c"): -1.0,
                model.metabolites.get_by_id("h_c"): 2.0,      # translated_note
                model.metabolites.get_by_id("nicrnt_c"): 1.0,
                model.metabolites.get_by_id("ppi_c"): 1.0,
                model.metabolites.get_by_id("co2_c"): 1.0
            }
            # translated_note, translated_note
            rxn.subtract_metabolites(rxn.metabolites)
            rxn.add_metabolites(correct_stoich)
            print(f"   translated_note {rxn.id} translated_note ID translated_note. ")

    if not found_nndpr:
        print("   translated_note: translated_note NNDPR translated_note!translated_note NAD translated_note. ")

    # 4. translated_note nadp_c translated_note
    print("\n translated_note NADP+ translated_note...")
    with model as m:
        # translated_note ATP, translated_note NaMN translated_note
        m.add_boundary(m.metabolites.get_by_id("atp_c"), type="sink").bounds = (0, 1000)
        m.objective = m.add_boundary(m.metabolites.get_by_id("nadp_c"), type="demand")
        flux = m.slim_optimize() or 0.0
        print(f" translated_note NADP+ translated_note: {flux:.4f}")

    cobra.io.save_json_model(model, out_file)
    print(f" translated_note: {out_file}")


# ---------------------------------------------------------------------
# Function: advanced_compartment_audit
# Purpose: Audit c/p/e connectivity and patch missing transport bridges.
# ---------------------------------------------------------------------
def advanced_compartment_audit(model_file="updated_consensus_purple.json"):
    print(">>> [FBA Agent] translated_note (Strict Suffix Audit)...")
    model = cobra.io.load_json_model(model_file)
    
    # translated_note
    VALID_COMPS = {'c', 'p', 'e', 'u'}
    
    # 1. translated_note: BaseID -> {Comp: FullID}
    # translated_note: translated_note re translated_note _c, _p translated_note
    mapping = {}
    for m in model.metabolites:
        match = re.search(r'^(.*)_([cpeu])$', m.id)
        if match:
            base_id, comp = match.groups()
            if base_id not in mapping:
                mapping[base_id] = {}
            mapping[base_id][comp] = m.id
        else:
            # translated_note(translated_note 12dgr181_9_c translated_note)
            continue

    broken_links = []

    # 2. translated_note: e <-> p <-> c
    # translated_note, c-p translated_note, p-e translated_note
    for base, comps in mapping.items():
        # A. translated_note c <-> p translated_note (translated_note)
        if 'c' in comps and 'p' in comps:
            m_c = model.metabolites.get_by_id(comps['c'])
            m_p = model.metabolites.get_by_id(comps['p'])
            if not set(m_c.reactions).intersection(set(m_p.reactions)):
                broken_links.append((base, 'c', 'p'))

        # B. translated_note p <-> e translated_note (translated_note)
        if 'p' in comps and 'e' in comps:
            m_p = model.metabolites.get_by_id(comps['p'])
            m_e = model.metabolites.get_by_id(comps['e'])
            if not set(m_p.reactions).intersection(set(m_e.reactions)):
                broken_links.append((base, 'p', 'e'))

    # 3. translated_note
    print(f"\n translated_note {len(broken_links)} translated_note: ")
    for base, c1, c2 in broken_links:
        # translated_note
        highlight = "  [CRITICAL]" if base in ['h', 'co2', 'atp', 'pi', 'adp'] else ""
        print(f"   {base:15} translated_note: {c1} <-> {c2}{highlight}")

    # 4. translated_note(translated_note)
    print("\n translated_note (translated_note h translated_note co2):")
    repair_logic = [
        ("CO2tpp", {"co2_p": -1, "co2_c": 1}, "CO2 translated_note"),
        ("CO2t",   {"co2_e": -1, "co2_p": 1}, "CO2 translated_note"),
        ("Htp",    {"h_p": -1, "h_c": 1},     "translated_note (translated_note)")
    ]
    for r_id, stoich, name in repair_logic:
        print(f"  # translated_note {name}:\n  # rxn = cobra.Reaction('{r_id}'); rxn.add_metabolites({stoich}); model.add_reactions([rxn])")


## Phase 3: Species-Specific Reconstruction (Inject Biological Identity)


In [ ]:
# ---------------------------------------------------------------------
# Function: inject_purple_photosynthesis
# Purpose: Inject purple-bacteria photosystem and remove incompatible legacy modules.
# ---------------------------------------------------------------------
def inject_purple_photosynthesis(model_file="updated_consensus_v3.json", out_file="updated_consensus_purple.json"):
    print(">>> [FBA Agent] translated_note (Purple Bacteria) translated_note...")
    model = cobra.io.load_json_model(model_file)

    # 1. translated_note: translated_note (Exchange)
    blocked_count = 0
    for rxn in model.exchanges:
        if 'photon' in rxn.id.lower() or 'light' in rxn.id.lower():
            rxn.bounds = (0, 0)
            blocked_count += 1
    print(f"   translated_note {blocked_count} translated_note/translated_note. ")

    # 2. translated_note (Cytochrome c2, translated_note _p) translated_note
    metabolites_to_add = [
        cobra.Metabolite("photon_purple_e", name="Photon (Purple RC)", compartment="e"),
        cobra.Metabolite("cytc2ox_p", name="Cytochrome c2 (oxidized)", compartment="p"),
        cobra.Metabolite("cytc2rd_p", name="Cytochrome c2 (reduced)", compartment="p")
    ]
    # translated_note try-except translated_note
    for m in metabolites_to_add:
        if m.id not in model.metabolites:
            model.add_metabolites([m])

    # translated_note (translated_note pq_c translated_note)
    pq_c = model.metabolites.get_by_id("pq_c")
    pqh2_c = model.metabolites.get_by_id("pqh2_c")
    h_c = model.metabolites.get_by_id("h_c")
    h_p = model.metabolites.get_by_id("h_p")

    new_rxns = []

    # 3. translated_note (Cyclic Electron Flow)
    
    # A. translated_note
    ex_photon = cobra.Reaction("EX_photon_purple_e", name="Purple Photon Exchange")
    ex_photon.add_metabolites({model.metabolites.get_by_id("photon_purple_e"): -1})
    ex_photon.bounds = (-100, 0)
    new_rxns.append(ex_photon)

    # B. IItranslated_note (Reaction Center)
    # translated_note, translated_note Cyt-c2 translated_note (translated_note)
    rc = cobra.Reaction("PURPLE_RC", name="Anoxygenic Reaction Center Type II")
    rc.add_metabolites({
        model.metabolites.get_by_id("photon_purple_e"): -2,
        model.metabolites.get_by_id("cytc2rd_p"): -2,
        pq_c: -1,
        h_c: -2,
        model.metabolites.get_by_id("cytc2ox_p"): 2,
        pqh2_c: 1
    })
    rc.bounds = (0, 1000)
    new_rxns.append(rc)

    # C. translated_note bc1 translated_note (Cytochrome bc1 Complex)
    # translated_note, translated_note Cyt-c2 translated_note. translated_note Q-cycle translated_note (translated_note PMF)
    bc1 = cobra.Reaction("PURPLE_CYTBC1", name="Cytochrome bc1 complex")
    bc1.add_metabolites({
        pqh2_c: -1,
        model.metabolites.get_by_id("cytc2ox_p"): -2,
        h_c: -2,
        pq_c: 1,
        model.metabolites.get_by_id("cytc2rd_p"): 2,
        h_p: 4
    })
    bc1.bounds = (0, 1000)
    new_rxns.append(bc1)

    model.add_reactions(new_rxns)
    print("   translated_note (RC) translated_note bc1(Cyt-bc1) translated_note. ")

    # 4. FBA translated_note
    print("\n translated_note...")
    solution = model.optimize()
    print(f"   translated_note: {solution.objective_value:.4f} h^-1")

    cobra.io.save_json_model(model, out_file)
    print(f"   translated_note: {out_file}")


# ---------------------------------------------------------------------
# Function: inject_curated_transports
# Purpose: Inject curated species-specific transport reactions from manual evidence.
# ---------------------------------------------------------------------
def inject_curated_transports(model_file="purple_bacteria.json", out_file="purple_bacteriav2.json"):
    print(">>> [FBA Agent] translated_note (Curated) translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    # translated_note: (translated_note Base ID, translated_note ID, translated_note)
    curated_list = [
        # translated_note
        ("ca2", "CA2t2pp", "p_to_c"),
        ("for", "FORt2pp", "p_to_c"),
        ("h2", "H2tex", "e_to_p"),
        ("so3", "SO3tex", "e_to_p"),
        ("tsul", "TSULtex", "e_to_p"),
        
        # translated_note (translated_note, translated_note, translated_note)
        ("fru", "FRUt2pp", "p_to_c"), 
        ("man", "MANtex", "e_to_p"),
        ("man", "MANt2pp", "p_to_c"),
        ("glyc", "GLYCtex", "e_to_p"),
        ("etoh", "ETOHtex", "e_to_p"),
        ("ppa", "PPAtex", "e_to_p"),
        
        # translated_note (translated_note)
        ("ala__L", "ALA__Ltex", "e_to_p"),
        ("ala__D", "ALA__Dtex", "e_to_p"),
        ("ile__L", "ILE__Ltex", "e_to_p"),
        ("lys__L", "LYS__Ltex", "e_to_p"),
        ("pro__L", "PRO__Ltex", "e_to_p"),
        ("thr__L", "THR__Ltex", "e_to_p"),
        ("val__L", "VAL__Ltex", "e_to_p"),
        ("asn__L", "ASN__Ltex", "e_to_p"),
        ("asn__L", "ASN__Lt2pp", "p_to_c"),
        ("glu__L", "GLU__Ltex", "e_to_p"),
        
        # translated_note (translated_note)
        ("adn", "ADNtex", "e_to_p"),
        ("adn", "ADNt2pp", "p_to_c"),
        ("cytd", "CYTDt2pp", "p_to_c"),
        ("dcyt", "DCYTtex", "e_to_p"),
        ("dcyt", "DCYTt2pp", "p_to_c"),
        ("dad_2", "DAD_2tex", "e_to_p"),
        ("dad_2", "DAD_2t2pp", "p_to_c"),
        ("dgsn", "DGSNt2pp", "p_to_c")
    ]
    
    new_reactions = []
    
    for base_id, rxn_id, path in curated_list:
        # translated_note
        if rxn_id in model.reactions:
            print(f"   translated_note: {rxn_id} (translated_note)")
            continue
            
        new_rxn = cobra.Reaction(rxn_id)
        new_rxn.name = f"{base_id} transport ({path.replace('_to_', ' to ')})"
        new_rxn.bounds = (-1000.0, 1000.0) # translated_note, translated_note
        
        try:
            if path == "e_to_p":
                met_e = model.metabolites.get_by_id(f"{base_id}_e")
                met_p = model.metabolites.get_by_id(f"{base_id}_p")
                new_rxn.add_metabolites({met_e: -1.0, met_p: 1.0})
            elif path == "p_to_c":
                met_p = model.metabolites.get_by_id(f"{base_id}_p")
                met_c = model.metabolites.get_by_id(f"{base_id}_c")
                new_rxn.add_metabolites({met_p: -1.0, met_c: 1.0})
                
            new_reactions.append(new_rxn)
            print(f"   translated_note: {rxn_id:12} | translated_note: {base_id:8} | translated_note: {path}")
            
        except KeyError as e:
            print(f"   translated_note: translated_note {e} (translated_note {rxn_id})")

    if new_reactions:
        model.add_reactions(new_reactions)
        cobra.io.save_json_model(model, out_file)
        print(f"\n================ translated_note ================")
        print(f" translated_note {len(new_reactions)} translated_note!")
        print(f" translated_note: {out_file}")
    else:
        print("\n translated_note, translated_note. ")


## Phase 4: Final Bottleneck Tracing & Gap-filling (Bring the Cell to Life)


In [ ]:
# ---------------------------------------------------------------------
# Function: test_all_open_exchanges
# Purpose: Run all-open exchange survivability test.
# ---------------------------------------------------------------------
def test_all_open_exchanges(model_file="updated_consensus_purple.json"):
    print(">>> [FBA Agent] translated_note Exchange translated_note (translated_note)...")
    model = cobra.io.load_json_model(model_file)
    
    # translated_note
    opened_list = []

    # 1. translated_note
    for rxn in model.exchanges:
        # translated_note(lb=0)translated_note
        if rxn.lower_bound == 0:
            rxn.lower_bound = -1000.0
            opened_list.append(f"{rxn.id} ({rxn.name})")
        else:
            # translated_note(translated_note)
            rxn.lower_bound = -1000.0

    # 2. translated_note
    print(f"\n translated_note {len(model.exchanges)} translated_note. ")
    print("--- translated_note (translated_note) ---")
    for item in opened_list[:20]: # translated_note20translated_note
        print(f"  + {item}")
    if len(opened_list) > 20: print(f"  ... translated_note {len(opened_list)-20} translated_note")

    # 3. translated_note FBA
    print("\n translated_note FBA translated_note...")
    solution = model.optimize()
    
    print("\n================ FBA translated_note ================")
    print(f" translated_note: {solution.status}")
    print(f" translated_note (Objective): {solution.objective_value:.4f} h^-1")
    print("==============================================\n")

    if solution.objective_value < 1e-6:
        print(" translated_note: translated_notetranslated_notetranslated_note, Biomass translated_note 0!")
        print("   translated_note ID translated_note(translated_note quln_c vs qns_c)translated_note. ")
    else:
        print(" translated_note!translated_note. ")
        print("   translated_note, translated_note. ")
        
        # translated_note
        print("\n translated_note (Flux < 0):")
        for rxn in model.exchanges:
            if solution.fluxes[rxn.id] < -1e-6:
                print(f"   translated_note: {rxn.id} | translated_note: {abs(solution.fluxes[rxn.id]):.4f}")


# ---------------------------------------------------------------------
# Function: deep_dive_biomass_components
# Purpose: Dissect biomass pack components using demand-based stress tests.
# ---------------------------------------------------------------------
def deep_dive_biomass_components(model_file="updated_consensus_purple.json"):
    print(">>> [FBA Agent] translated_note Biomass translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    # 1. translated_note & translated_note (translated_note)
    for ex in model.exchanges: ex.lower_bound = -1000.0
    magic = cobra.Reaction("MAGIC_COFACTORS")
    magic.add_metabolites({
        model.metabolites.get_by_id("adp_c"): -1,
        model.metabolites.get_by_id("pi_c"): -1,
        model.metabolites.get_by_id("atp_c"): 1,
        model.metabolites.get_by_id("h2o_c"): 1
    })
    magic.bounds = (0, 1000)
    model.add_reactions([magic])

    # 2. translated_notetranslated_notetranslated_note ID
    target_rxn_ids = ["BIOMASS_COFACTORS", "BIOMASS_MISC", "BIOMASS_PIGMENTS"]
    
    total_broken = 0

    # 3. translated_note
    for rxn_id in target_rxn_ids:
        try:
            rxn = model.reactions.get_by_id(rxn_id)
            print(f"\n================ translated_note: {rxn_id} ({rxn.name}) ================")
            
            # translated_note 0 translated_note(translated_note)
            reactants = [met for met, coeff in rxn.metabolites.items() if coeff < 0]
            
            broken_in_this_bag = []
            
            for met in reactants:
                with model as m:
                    dm = m.add_boundary(met, type="demand")
                    m.objective = dm
                    flux = m.slim_optimize() or 0.0
                    
                    if flux < 1e-6:
                        broken_in_this_bag.append(met.id)
                        print(f"   translated_note: {met.id:15} | {met.name}")
                    else:
                        print(f"   translated_note: {met.id}")
            
            total_broken += len(broken_in_this_bag)
            
        except KeyError:
            print(f"\n translated_note: translated_note {rxn_id}!")

    print(f"\n================ translated_note ================")
    if total_broken == 0:
        print(" translated_note!translated_note!(translated_note, translated_note)")
    else:
        print(f" translated_note {total_broken} translated_note!")
        print("translated_note: translated_note  translated_note, translated_note. ")


# ---------------------------------------------------------------------
# Function: auto_gapfill_model
# Purpose: Perform automatic gap-filling against a pan-model reaction pool.
# ---------------------------------------------------------------------
def auto_gapfill_model(broken_model_file="updated_consensus_v3.json", 
                       pan_models_dir="bigg_resources/models",
                       out_model_file="updated_consensus_v4.json"):
    print(">>> [FBA Agent] translated_note (GapFilling) translated_note...")
    
    # 1. translated_note
    try:
        model = cobra.io.load_json_model(broken_model_file)
    except Exception as e:
        print(f" translated_note: {e}")
        return

    # 2. translated_notetranslated_note (Universal Model)
    print("\n translated_note (Universal Model), translated_note...")
    universal = cobra.Model("Universal_Pan_Model")
    existing_rxn_ids = {r.id for r in model.reactions}
    
    model_files = glob.glob(os.path.join(pan_models_dir, '*.json'))
    added_rxns = 0
    for m_file in model_files:
        try:
            ref_model = cobra.io.load_json_model(m_file)
            # translated_note, translated_note
            for rxn in ref_model.reactions:
                if rxn.id not in existing_rxn_ids and rxn.id not in universal.reactions:
                    # translated_note, translated_note
                    new_rxn = rxn.copy()
                    universal.add_reactions([new_rxn])
                    added_rxns += 1
        except Exception as e:
            continue
            
    print(f"   translated_note!translated_note {added_rxns} translated_note. ")

    # 3. translated_note GapFiller translated_note
    print("\n translated_note (translated_note, translated_note)...")
    try:
        # translated_note objective (translated_note Biomass)
        # lower_bound=0.01 translated_note Biomass translated_note
        gapfiller = gapfilling.GapFiller(model, universal=universal, demand_reactions=False, lower_bound=0.01)
        
        # translated_note
        solutions = gapfiller.fill()
        best_solution = solutions[0]
        
        print("\n translated_note!GapFiller translated_note!")
        print(f"  translated_note {len(best_solution)} translated_note:")
        
        for rxn in best_solution:
            print(f"     {rxn.id}: {rxn.name} | {rxn.reaction}")
            # translated_note
            model.add_reactions([rxn.copy()])
            
        # 4. translated_note
        print("\n translated_note FBA translated_note...")
        fba_res = model.optimize()
        print(f"   translated_note: {fba_res.objective_value:.4f} h^-1")
        
        # translated_note
        cobra.io.save_json_model(model, out_model_file)
        print(f"\n translated_note!translated_note: {out_model_file}")

    except RuntimeError:
        print("\n GapFiller translated_note: translated_note, translated_note!")
        print("   translated_note: translated_note, translated_note(Exchange)translated_note. ")


# ---------------------------------------------------------------------
# Function: test_final_seven_parts
# Purpose: Retest known blacklist metabolites after pathway fixes.
# ---------------------------------------------------------------------
def test_final_seven_parts(model_file="updated_consensus_purple.json"): # translated_note
    print(">>> [FBA Agent] translated_note 7 translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    # translated_note
    for ex in model.exchanges: ex.lower_bound = -1000.0
    
    # translated_note
    targets = [
        'adocbl_c', 'pheme_c', 'sheme_c', 'bcpa_c', 'bcpb_c',  # translated_note (translated_note uppg3_c translated_note)
        '10fthf_c', # translated_note
        'pydx5p_c'  # translated_note B6
    ]
    
    print("\n translated_note: ")
    success_count = 0
    for met_id in targets:
        try:
            met = model.metabolites.get_by_id(met_id)
            with model as m:
                m.objective = m.add_boundary(met, type="demand")
                flux = m.slim_optimize() or 0.0
                
                if flux > 1e-6:
                    print(f"   translated_note: {met_id:15}")
                    success_count += 1
                else:
                    print(f"   translated_note: {met_id:15}")
        except KeyError:
             print(f"   translated_note: {met_id:15}")
             
    print("\n================ translated_note ================")
    if success_count >= 5:
        print(" translated_note!translated_note!")
    elif success_count == 0:
        print(" translated_note!")
        print(" translated_note ID translated_note: translated_note `uropgen3_c`, translated_note `uppg3_c`. ")
        print("   translated_note: translated_note `uppg3_c` translated_note, translated_note; translated_note, translated_note `uppg3_c` translated_note. ")


# ---------------------------------------------------------------------
# Function: final_two_patches
# Purpose: Apply final manual patches and validate growth in non-magic conditions.
# ---------------------------------------------------------------------
def final_two_patches(model_file="updated_consensus_purple.json", out_file="purple_bacteria_FINAL.json"):
    print(">>> [FBA Agent] translated_note 2 translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    new_rxns = []

    # ==========================================
    # translated_note 1: translated_note (10fthf_c)
    # ==========================================
    if "FTHFLi" not in model.reactions:
        rxn1 = cobra.Reaction("FTHFLi")
        rxn1.name = "Formate--tetrahydrofolate ligase"
        # atp + formate + thf -> 10-formyl-thf + adp + pi
        rxn1.add_metabolites({
            model.metabolites.get_by_id("atp_c"): -1,
            model.metabolites.get_by_id("for_c"): -1,
            model.metabolites.get_by_id("thf_c"): -1,
            model.metabolites.get_by_id("10fthf_c"): 1,
            model.metabolites.get_by_id("adp_c"): 1,
            model.metabolites.get_by_id("pi_c"): 1
        })
        rxn1.bounds = (0, 1000)
        new_rxns.append(rxn1)
        print("   translated_note: FTHFLi (10fthf_c translated_note)")

    # ==========================================
    # translated_note 2: translated_note B6 / PLP (pydx5p_c)
    # ==========================================
    # translated_note PdxST translated_note (R5P + G3P + Gln -> PLP)
    if "PDXSP" not in model.reactions:
        rxn2 = cobra.Reaction("PDXSP")
        rxn2.name = "Pyridoxal 5'-phosphate synthase (glutamine hydrolyzing)"
        rxn2.add_metabolites({
            model.metabolites.get_by_id("ru5p__D_c"): -1,
            model.metabolites.get_by_id("g3p_c"): -1,     
            model.metabolites.get_by_id("gln__L_c"): -1,
            model.metabolites.get_by_id("pydx5p_c"): 1,
            model.metabolites.get_by_id("glu__L_c"): 1,
            model.metabolites.get_by_id("pi_c"): 1,
            model.metabolites.get_by_id("h2o_c"): 3,
            model.metabolites.get_by_id("h_c"): 1
        })
        rxn2.bounds = (0, 1000)
        
        # translated_note: translated_note R5P translated_note r5p_c, translated_note
        if "r5p_c" in model.metabolites:
            rxn2.subtract_metabolites({model.metabolites.get_by_id("ru5p__D_c"): -1})
            rxn2.add_metabolites({model.metabolites.get_by_id("r5p_c"): -1})
            
        new_rxns.append(rxn2)
        print("   translated_note: PDXSP (pydx5p_c translated_note)")

    model.add_reactions(new_rxns)

    # ==========================================
    # translated_note FBA translated_note (translated_note Biomass translated_note)
    # ==========================================
    print("\n translated_note Medium translated_note Biomass translated_note...")
    for ex in model.exchanges: 
        ex.lower_bound = -1000.0
        
    # translated_note, translated_note100%translated_note!
    if "MAGIC_COFACTORS" in model.reactions: model.remove_reactions(["MAGIC_COFACTORS"])
    if "MAGIC_ATP" in model.reactions: model.remove_reactions(["MAGIC_ATP"])
        
    # translated_note Biomass (translated_note BIOMASS__1)
    # translated_note objective
    solution = model.optimize()
    
    print("\n================ FBA translated_note ================")
    print(f" translated_note (Objective): {solution.objective_value:.4f} h^-1")
    print("==============================================\n")
    
    if solution.objective_value > 1e-6:
        cobra.io.save_json_model(model, out_file)
        print(f" translated_note!translated_note, translated_note!translated_note: {out_file}")
    else:
        print(" translated_note 0, translated_note test_final_seven_parts translated_note, translated_note ID translated_note. ")


## Phase 5: Phenotype Validation & Application Simulations (Top-Down Analysis)
Source: File 1 + File 2

In [ ]:
# ---------------------------------------------------------------------
# Function: find_essential_medium_by_fva
# Purpose: Infer minimal medium constraints with FVA.
# ---------------------------------------------------------------------
def find_essential_medium_by_fva(model_file="purple_bacteria.json"):
    print(">>> [FBA Agent] translated_note FVA translated_note: translated_note...")
    model = cobra.io.load_json_model(model_file)

    # 1. translated_note Exchange 
    for ex in model.exchanges:
        ex.lower_bound = -100.0
        
    # 2. translated_note Biomass translated_note 
    biomass_rxns = cobra.util.solver.linear_reaction_coefficients(model)
    if not biomass_rxns:
        biomass_rxn = [r for r in model.reactions if 'biomass' in r.id.lower()][0]
    else:
        biomass_rxn = list(biomass_rxns.keys())[0]

    # translated_note
    target_growth = 5
    biomass_rxn.lower_bound = target_growth
    biomass_rxn.upper_bound = target_growth
    print(f"translated_note Biomass translated_note: {target_growth} h^-1")

    # 3. translated_note Exchange translated_note FVA
    print("translated_noteFVA translated_note...")
    ex_rxns = model.exchanges
    # translated_note FVA translated_note 0.1 translated_note
    fva_res = flux_variability_analysis(model, reaction_list=ex_rxns, loopless=False)

    # 4. translated_note FVA translated_note, translated_note
    essential_nutrients = []
    
    print("\n================ translated_note (Minimal Medium) ================")
    for rxn in ex_rxns:
        # translated_note: translated_note CobraPy translated_note, translated_note
        # translated_note maximum translated_note, translated_note
        max_flux = fva_res.loc[rxn.id, "maximum"]
        min_flux = fva_res.loc[rxn.id, "minimum"]
        
        if max_flux < -1e-6:
            # translated_note (translated_note FVA translated_note maximum translated_note)
            required_amount = abs(max_flux)
            essential_nutrients.append((rxn.id, required_amount, rxn.name))
            print(f"{rxn.id:18} | translated_note: {required_amount:8.10f} | {rxn.name}")


# ---------------------------------------------------------------------
# Function: in_silico_phenotype_microarray
# Purpose: Simulate in silico phenotype arrays across carbon/nitrogen sources.
# ---------------------------------------------------------------------
def in_silico_phenotype_microarray(model_file="purple_bacteriav2.json"):
    print(">>> [FBA Agent] translated_note (In Silico Biolog Plate) translated_note...")
    model = cobra.io.load_json_model(model_file)
    
    # 1. translated_note Minimal Medium (translated_note, translated_note, translated_note)
    for ex in model.exchanges: ex.lower_bound = 0.0 # translated_note
    essential_ions = [
        'EX_cobalt2_e', 'EX_zn2_e', 'EX_so4_e', 'EX_ca2_e', 'EX_mn2_e', 
        'EX_mg2_e', 'EX_cu2_e', 'EX_k_e', 'EX_fe3_e', 'EX_mobd_e', 
        'EX_na1_e', 'EX_cl_e', 'EX_bo3_e', 'EX_pi_e', 'EX_h2o_e', 'EX_h_e'
    ]
    for ion in essential_ions:
        if ion in model.reactions: model.reactions.get_by_id(ion).lower_bound = -1000.0
    model.reactions.get_by_id("EX_photon_purple_e").lower_bound = -2000.0
    if "EX_for_e" in model.reactions: model.reactions.get_by_id("EX_for_e").lower_bound = -1.0
    # 2. translated_note 
    carbon_sources = {
        "EX_co2_e": "CO2 (translated_note)",
        "EX_for_e": "Formate (translated_note)",
        "EX_glc__D_e": "D-Glucose (translated_note)",
        "EX_fru_e": "Fructose (translated_note)",
        "EX_sucr_e": "Sucrose (translated_note)",
        "EX_ac_e": "Acetate (translated_note)",
        "EX_mal__L_e": "L-Malate (translated_note)",
        "EX_succ_e": "Succinate (translated_note)",
        "EX_glyc_e": "Glycerol (translated_note)",
        "EX_pyr_e": "Pyruvate (translated_note)"
    }
    
    nitrogen_sources = {
        "EX_nh4_e": "Ammonia (translated_note)",
        "EX_no3_e": "Nitrate (translated_note)",
        "EX_no2_e": "Nitrite (translated_note)",
        "EX_n2_e": "Dinitrogen (translated_note)",
        "EX_urea_e": "Urea (translated_note)",
        "EX_glu__L_e": "L-Glutamate "
    }

    print("\n================  translated_note ================")
    if "EX_nh4_e" in model.reactions: model.reactions.get_by_id("EX_nh4_e").lower_bound = -100.0
    
    for ex_id, name in carbon_sources.items():
        if ex_id in model.reactions:
            with model as m: # translated_note with translated_note, translated_note
                m.reactions.get_by_id(ex_id).lower_bound = -20.0 # translated_note
                sol = m.slim_optimize()
                status = " translated_note" if sol > 1e-4 else " translated_note"
                print(f"  {status} | translated_note: {sol:6.4f} | {name}")
        else:
            print(f"   translated_note {name} translated_note Exchange translated_note")

    print("\n================  translated_note ================")
    if "EX_nh4_e" in model.reactions: model.reactions.get_by_id("EX_nh4_e").lower_bound = 0.0 # translated_note
    if "EX_mal__L_e" in model.reactions: model.reactions.get_by_id("EX_mal__L_e").lower_bound = -20.0
    
    for ex_id, name in nitrogen_sources.items():
        if ex_id in model.reactions:
            with model as m:
                m.reactions.get_by_id(ex_id).lower_bound = -20.0 # translated_note
                sol = m.slim_optimize()
                status = " translated_note" if sol > 1e-4 else " translated_note"
                print(f"  {status} | translated_note: {sol:6.4f} | {name}")
        else:
            print(f"   translated_note {name} translated_note Exchange translated_note")


## Suggested Execution Skeleton (Manual Control)

Uncomment and execute step-by-step when ready. Keep checkpoints between phases.

In [ ]:
# ------------------------------
# Manual-review-first mode (recommended rollback state)
# Baseline model: Models/DSM123.json
# Working copy:   Models/DSM123_manual_working.json
# ------------------------------
# Step 0: Ensure a clean editable working copy
# import shutil; shutil.copy2('Models/DSM123.json', 'Models/DSM123_manual_working.json')

# Step 1: Generate evidence tables only (do not auto-apply full replacement)
# create_fasta_from_gbff(gbff_file='genomes/Rpal_BisA53.gb', fasta_out='prots/Rpal_ref.fasta')
# run_bbh_pipeline(ref_fasta='prots/Rpal_ref.fasta', target_fasta='prots/DSM123.fa')
# audit_model_reactions(identity_thresh=50.0, evalue_thresh=1e-2)

# Step 2: Manually inspect outputs and curate reaction-level edits
# - model_gpr_conflicts_report.csv
# - model_missing_genes_candidates.csv
# - gpr_manual_review_required.csv

# Step 3: Apply a small batch of manual edits, then run one checkpoint test
# run_fba_check(model_file='Models/DSM123_manual_working.json')
# check_metabolite_consistency(model_file='Models/DSM123_manual_working.json')

# Step 4: Iterate (edit -> test -> inspect) until growth/connectivity improves
# test_all_open_exchanges(model_file='Models/DSM123_manual_working.json')
# deep_dive_biomass_components(model_file='Models/DSM123_manual_working.json')

# Step 5: Run phenotype analyses only after model is stable
# find_essential_medium_by_fva(model_file='Models/DSM123_manual_working.json')
# in_silico_phenotype_microarray(model_file='Models/DSM123_manual_working.json')
